In [28]:
# ══════════════════════════════════════════════════════════════
# CELL 1: Setup and GPU check
# ══════════════════════════════════════════════════════════════
!pip install -q transformers

import pandas as pd
import numpy as np
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, roc_curve)
from sklearn.model_selection import StratifiedShuffleSplit
import matplotlib.pyplot as plt
import json
import os
import shutil

In [29]:
# Verify GPU
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Save directory — everything goes here
save_dir = Path("/kaggle/working")
print(f"\nSave directory: {save_dir}")

CUDA available: True
GPU: Tesla T4
Device: cuda

Save directory: /kaggle/working


In [30]:
# ══════════════════════════════════════════════════════════════
# CELL 2: Load data
# ══════════════════════════════════════════════════════════════
train_df = pd.read_csv(
    "/kaggle/input/datasets/rheapandita/grids-newdatasplits/train_df.csv"
)
test_df = pd.read_csv(
    "/kaggle/input/datasets/rheapandita/grids-newdatasplits/test_df.csv"
)
val_df = pd.read_csv(
    "/kaggle/input/datasets/rheapandita/grids-newdatasplits/val_df.csv"
)

print(f"train_df shape: {train_df.shape}")
print(f"test_df shape:  {test_df.shape}")
print(f"clinical_text present: {'clinical_text' in train_df.columns}")
print(f"Train positive rate: "
      f"{train_df['respiratory_failure'].mean()*100:.1f}%")
print(f"Test positive rate:  "
      f"{test_df['respiratory_failure'].mean()*100:.1f}%")

train_df shape: (63852, 183)
test_df shape:  (13769, 183)
clinical_text present: True
Train positive rate: 38.0%
Test positive rate:  38.2%


In [31]:
# ══════════════════════════════════════════════════════════════
# CELL 3: Training configuration
# ══════════════════════════════════════════════════════════════
MODEL_NAME        = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LEN           = 512
BATCH_SIZE        = 16
EPOCHS            = 5
LR                = 2e-5
TRAIN_SAMPLE      = 10_000
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [32]:
# Stratified subsample for train (keeps class balance)
train_sample = train_df.groupby("respiratory_failure", group_keys=False).apply(
    lambda x: x.sample(min(len(x), TRAIN_SAMPLE // 2), random_state=42)
).reset_index(drop=True)

/tmp/ipykernel_55/2339037432.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_sample = train_df.groupby("respiratory_failure", group_keys=False).apply(


In [33]:
# ── Tokenizer ─────────────────────────────────────────────────
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded ✓")


Loading tokenizer...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizer loaded ✓


In [ ]:
# ── Dataset class ─────────────────────────────────────────────
class ClinicalTextDataset(Dataset):
    def __init__(self, df):
        self.texts  = df["clinical_text"].tolist()
        self.labels = df["respiratory_failure"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return (
            {k: v.squeeze(0) for k, v in enc.items()},
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

print("ClinicalTextDataset defined ✓")


In [ ]:
# ── Create datasets ───────────────────────────────────────────
train_dataset = ClinicalTextDataset(train_sample)
val_dataset   = ClinicalTextDataset(val_df)
test_dataset  = ClinicalTextDataset(test_df)

print(f"Train dataset: {len(train_dataset):,} samples")
print(f"Val dataset:   {len(val_dataset):,} samples")
print(f"Test dataset:  {len(test_dataset):,} samples")
print("Datasets created ✓")


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,   shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader):,}")
print(f"Val batches:   {len(val_loader):,}")
print(f"Test batches:  {len(test_loader):,}")
print("DataLoaders created ✓")


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# ── Model ─────────────────────────────────────────────────────
print("\nLoading ClinicalBERT...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(device)
print("Model loaded ✓")

# ── Class weights ─────────────────────────────────────────────
cw      = compute_class_weight("balanced",
            classes=np.array([0, 1]),
            y=train_sample["respiratory_failure"].values)
weights = torch.tensor(cw, dtype=torch.float).to(device)
loss_fn   = torch.nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

print(f"Class weights — Negative: {cw[0]:.3f} | Positive: {cw[1]:.3f}")
print("Model and optimizer ready ✓")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4: Training loop — val set for epoch selection
# ══════════════════════════════════════════════════════════════
print(f"\nStarting training ({EPOCHS} epochs)...")
print("=" * 55)

train_losses = []
val_aurocs   = []
best_val_auc = 0
best_epoch   = 0

def run_eval(loader):
    model.eval()
    probs, labels_out = [], []
    with torch.no_grad():
        for enc_dict, label_tensor in loader:
            enc_dict = {k: v.to(device) for k, v in enc_dict.items()}
            outputs  = model(**enc_dict)
            p = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
            probs.extend(p)
            labels_out.extend(label_tensor.numpy())
    return roc_auc_score(labels_out, probs), np.array(probs), np.array(labels_out)

for epoch in range(EPOCHS):

    # ── Train ─────────────────────────────────────────────────
    model.train()
    total_loss, n_batches = 0, 0

    for i, (enc_dict, label_tensor) in enumerate(train_loader):
        enc_dict     = {k: v.to(device) for k, v in enc_dict.items()}
        label_tensor = label_tensor.to(device)

        optimizer.zero_grad()
        outputs = model(**enc_dict)
        loss    = loss_fn(outputs.logits, label_tensor)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

        if (i + 1) % 50 == 0:
            print(f"  Epoch {epoch+1} | Batch {i+1}/{len(train_loader)} | "
                  f"Loss: {total_loss/n_batches:.4f}")

    avg_loss = total_loss / n_batches
    train_losses.append(avg_loss)

    # ── Evaluate on VAL (used for model selection only) ───────
    val_auc, _, _ = run_eval(val_loader)
    val_aurocs.append(val_auc)

    print(f"\nEpoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val AUROC: {val_auc:.4f}")

    # Save every epoch
    epoch_dir = save_dir / f"clinicalbert_epoch{epoch+1}"
    epoch_dir.mkdir(exist_ok=True)
    model.save_pretrained(str(epoch_dir))
    tokenizer.save_pretrained(str(epoch_dir))

    # Track best by val AUROC (NOT test)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch   = epoch + 1
        best_dir     = save_dir / "clinicalbert_best"
        if best_dir.exists():
            shutil.rmtree(best_dir)
        shutil.copytree(str(epoch_dir), str(best_dir))
        print(f"  ✓ New best val model saved (epoch {epoch+1}, val AUROC={val_auc:.4f})")

    results_so_far = {
        "completed_epochs": epoch + 1,
        "train_losses":     train_losses,
        "val_aurocs":       val_aurocs,
        "best_epoch":       best_epoch,
        "best_val_auroc":   best_val_auc,
    }
    with open(save_dir / "results_so_far.json", "w") as f:
        json.dump(results_so_far, f, indent=2)

print("\n" + "=" * 55)
print(f"Training complete ✓  Best epoch: {best_epoch} (Val AUROC={best_val_auc:.4f})")
print("TEST SET UNTOUCHED — run Cell 5 only once for final evaluation")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5: Final evaluation on held-out TEST SET (run once only)
# ══════════════════════════════════════════════════════════════
# Load best model (selected by val AUROC)
model = AutoModelForSequenceClassification.from_pretrained(
    str(save_dir / "clinicalbert_best")
).to(device)

test_auc, all_probs, all_labels = run_eval(test_loader)
preds = (np.array(all_probs) > 0.5).astype(int)

print(f"Best epoch (by val):  {best_epoch}  |  Val AUROC:  {best_val_auc:.4f}")
print(f"Final TEST AUROC:     {test_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(all_labels, preds,
      target_names=["Stable", "Respiratory Failure"]))
print(f"\nConfusion Matrix:")
print(confusion_matrix(all_labels, preds))

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr,
         label=f"ClinicalBERT (AUC={test_auc:.3f})",
         color="steelblue", linewidth=2)
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ClinicalBERT ROC — ICU Respiratory Failure (held-out test)")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(save_dir / "roc_curve_clinicalbert.png", dpi=150)
plt.show()

# Save final results
final_results = {
    "model":            MODEL_NAME,
    "train_sample":     TRAIN_SAMPLE,
    "val_size":         len(val_df),
    "test_size":        len(test_df),
    "epochs_trained":   EPOCHS,
    "best_epoch":       best_epoch,
    "best_val_auroc":   best_val_auc,
    "final_test_auroc": test_auc,
    "train_losses":     train_losses,
    "val_aurocs":       val_aurocs,
}
with open(save_dir / "final_results.json", "w") as f:
    json.dump(final_results, f, indent=2)
print("✓ final_results.json saved")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6: Rename best model to final and list all outputs
# ══════════════════════════════════════════════════════════════
final_dir = save_dir / "clinicalbert_final"
if final_dir.exists():
    shutil.rmtree(final_dir)
shutil.copytree(str(save_dir / "clinicalbert_best"), str(final_dir))

print(f"Best epoch: {best_epoch} (Val AUROC={val_aurocs[best_epoch-1]:.4f})")
print(f"Final model saved to: {final_dir}")

# List everything saved
print(f"\nAll files saved to /kaggle/working:")
for f in sorted(save_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1_000_000
        print(f"  {str(f).replace(str(save_dir)+'/', ''):<55} {size_mb:.1f} MB")

print("\n" + "="*55)
print("IMPORTANT: Click 'Save & Run All' or 'Commit' in the")
print("top right to permanently save all outputs to Kaggle.")
print("After committing, download files from Output tab.")
print("="*55)
